# 서울시 문화누리대상자 성·연령·장애별 인구 추정

행정동별 등록장애인 비율로 격자별 문화누리대상자 성·연령 인구를 장애인과 비장애인으로 배분합니다.

## 1. 분석 설계

- 분석 단위: 100m 격자 × 성별 × 연령대
- 장애인 비율: 2024년 행정동 등록장애인 수 / 행정동 총인구
- 배분 방법: 성·연령별 대상자 수에 행정동 장애인 비율 적용
- 정수화: 격자별 장애인 목표 인원을 최대 나머지 방식으로 배분
- 품질 기준: 결측·중복·음수 0, 기존 대상자 총량 보존

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "02_population_estimation":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
RAW = PROJECT_ROOT / "data" / "raw" / "population" / "demographics" / "source"
PROCESSED = PROJECT_ROOT / "data" / "processed" / "estimated_target_population"
PROCESSED.mkdir(parents=True, exist_ok=True)

AGE_SEX_FILE = PROCESSED / "서울시_격자_100m_문화누리대상자_성연령별_인구수.csv"
DISABLED_FILE = RAW / "서울시 장애인 현황(장애유형별_동별) 통계.csv"
DONG_POP_FILE = RAW / "서울시_행정동별_인구수_2024.csv"
OUTPUT_FILE = PROCESSED / "서울시_격자_100m_문화누리대상자_성연령장애별_인구수.csv"

for path in [AGE_SEX_FILE, DISABLED_FILE, DONG_POP_FILE]:
    assert path.exists(), f"입력 파일 없음: {path}"
print("입력 파일 확인 완료")

입력 파일 확인 완료


## 2. 데이터 불러오기 및 기본 점검

In [2]:
age_sex = pd.read_csv(AGE_SEX_FILE, encoding="utf-8-sig")
disabled_raw = pd.read_csv(DISABLED_FILE, encoding="cp949", dtype=str, low_memory=False)
dong_population = pd.read_csv(DONG_POP_FILE, encoding="utf-8-sig")
display(pd.DataFrame({
    "데이터": ["성·연령별 대상자", "장애인 현황", "행정동 총인구"],
    "행 수": [len(age_sex), len(disabled_raw), len(dong_population)],
}))
assert age_sex[["GRID_CD", "성별", "연령대"]].duplicated().sum() == 0
assert age_sex["문화누리대상자_성연령별_추정_인구수"].ge(0).all()

,데이터,행 수
0,성·연령별 대상자,1452672
1,장애인 현황,151452
2,행정동 총인구,426


## 3. 행정동별 장애인 비율 산출

- 2024년 자료만 사용
- 행정동이 아닌 `기타` 행 제외
- 마침표(`.`)로 표기된 7개 통합동을 기준 인구표의 가운데점(`·`)으로 통일
- 장애유형과 성별을 합산하여 행정동별 등록장애인 수 산출

In [3]:
disabled = disabled_raw.copy()
disabled["기준년"] = pd.to_numeric(disabled["기준년"], errors="coerce")
disabled = disabled.loc[disabled["기준년"].eq(2024) & disabled["읍면동"].ne("기타")].copy()
disabled["행정동"] = disabled["읍면동"].str.replace(".", "·", regex=False)
disabled["장애인수"] = pd.to_numeric(
    disabled["장애인수(명)"].str.replace(",", "", regex=False), errors="coerce"
).fillna(0)

disabled_dong = disabled.groupby(["시군구", "행정동"], as_index=False)["장애인수"].sum()
disabled_ratio = dong_population.merge(
    disabled_dong, on=["시군구", "행정동"], how="left", validate="one_to_one"
)
disabled_ratio["행정동_장애인비율"] = disabled_ratio["장애인수"] / disabled_ratio["행정동별_인구수"]

display(pd.Series({
    "행정동 수": len(disabled_ratio),
    "장애인수 결측 행정동": int(disabled_ratio["장애인수"].isna().sum()),
    "장애인비율 결측 행정동": int(disabled_ratio["행정동_장애인비율"].isna().sum()),
    "장애인비율 최솟값": disabled_ratio["행정동_장애인비율"].min(),
    "장애인비율 최댓값": disabled_ratio["행정동_장애인비율"].max(),
    "2024년 등록장애인 합계": int(disabled_dong["장애인수"].sum()),
}, name="확인값").to_frame())
assert disabled_ratio["장애인수"].isna().sum() == 0
assert disabled_ratio["행정동_장애인비율"].between(0, 1).all()

,확인값
행정동 수,426.000000
장애인수 결측 행정동,0.000000
장애인비율 결측 행정동,0.000000
장애인비율 최솟값,0.000000
장애인비율 최댓값,0.171938
2024년 등록장애인 합계,386314.000000


## 4. 성·연령별 대상자 장애 여부 배분

소수 추정치는 격자별 장애인 목표 총량을 보존하도록 최대 나머지 방식으로 정수화합니다.

In [4]:
result = age_sex.merge(
    disabled_ratio[["시군구", "행정동", "행정동_장애인비율"]],
    on=["시군구", "행정동"], how="left", validate="many_to_one"
)
missing_ratio = result.loc[result["행정동_장애인비율"].isna(), ["시군구", "행정동"]].drop_duplicates()
print("장애인비율 미연결 행정동 수:", len(missing_ratio))
if len(missing_ratio):
    display(missing_ratio)
assert len(missing_ratio) == 0

base_col = "문화누리대상자_성연령별_추정_인구수"
disabled_col = "문화누리대상자_성연령장애별_추정_인구수"
non_disabled_col = "문화누리대상자_성연령비장애별_추정_인구수"

result["장애인_raw"] = result[base_col] * result["행정동_장애인비율"]
result["장애인_내림"] = np.floor(result["장애인_raw"]).astype(int)
result["장애인_소수"] = result["장애인_raw"] - result["장애인_내림"]

grid_target = result.groupby("GRID_CD", as_index=False).agg(
    격자_대상자수=(base_col, "sum"),
    장애인비율=("행정동_장애인비율", "first"),
    장애인_내림합=("장애인_내림", "sum"),
)
grid_target["격자_장애인목표"] = np.rint(
    grid_target["격자_대상자수"] * grid_target["장애인비율"]
).astype(int)
grid_target["추가배분수"] = grid_target["격자_장애인목표"] - grid_target["장애인_내림합"]

result = result.merge(grid_target[["GRID_CD", "추가배분수"]], on="GRID_CD", validate="many_to_one")
result["추가배분순위"] = (
    result.sort_values(["GRID_CD", "장애인_소수"], ascending=[True, False])
    .groupby("GRID_CD").cumcount().add(1)
)
result[disabled_col] = result["장애인_내림"] + (
    result["추가배분순위"] <= result["추가배분수"]
).astype(int)
result[non_disabled_col] = result[base_col] - result[disabled_col]
result.head()

장애인비율 미연결 행정동 수: 0


,GRID_CD,시군구,행정동,성별,연령대,문화누리대상자_성연령별_추정_인구수,행정동_장애인비율,장애인_raw,장애인_내림,장애인_소수,추가배분수,추가배분순위,문화누리대상자_성연령장애별_추정_인구수,문화누리대상자_성연령비장애별_추정_인구수
0,다사603367,서초구,양재2동,남성,0-5세,0,0.031461,0.0,0,0.0,0,1,0,0
1,다사603367,서초구,양재2동,남성,100세-,0,0.031461,0.0,0,0.0,0,2,0,0
2,다사603367,서초구,양재2동,남성,15-19세,0,0.031461,0.0,0,0.0,0,3,0,0
3,다사603367,서초구,양재2동,남성,20-29세,0,0.031461,0.0,0,0.0,0,4,0,0
4,다사603367,서초구,양재2동,남성,30-39세,0,0.031461,0.0,0,0.0,0,5,0,0


## 5. 데이터 품질 점검

- 결측·중복·음수 여부 확인
- 장애인 + 비장애인 합계의 기존 대상자 총량 보존 확인
- 격자별 장애인 합계와 정수화 목표 비교

In [5]:
row_error = result[base_col] - result[disabled_col] - result[non_disabled_col]
grid_disabled = result.groupby("GRID_CD", as_index=False)[disabled_col].sum().merge(
    grid_target[["GRID_CD", "격자_장애인목표"]], on="GRID_CD", validate="one_to_one"
)
check_columns = [
    "GRID_CD", "시군구", "행정동", "성별", "연령대", "행정동_장애인비율",
    disabled_col, non_disabled_col,
]
quality = pd.Series({
    "최종 행 수": len(result),
    "격자 수": result["GRID_CD"].nunique(),
    "격자·성별·연령대 중복": int(result[["GRID_CD", "성별", "연령대"]].duplicated().sum()),
    "전체 결측": int(result[check_columns].isna().sum().sum()),
    "음수 인구": int(((result[disabled_col] < 0) | (result[non_disabled_col] < 0)).sum()),
    "행 단위 총량 오차": int(row_error.abs().sum()),
    "격자 장애인 목표 오차": int((grid_disabled[disabled_col] - grid_disabled["격자_장애인목표"]).abs().sum()),
    "전체 대상자 합계": int(result[base_col].sum()),
    "장애인 추정 합계": int(result[disabled_col].sum()),
    "비장애인 추정 합계": int(result[non_disabled_col].sum()),
}, name="확인값")
display(quality.to_frame())
for item in ["격자·성별·연령대 중복", "전체 결측", "음수 인구", "행 단위 총량 오차", "격자 장애인 목표 오차"]:
    assert quality[item] == 0

,확인값
최종 행 수,1452672
격자 수,60528
격자·성별·연령대 중복,0
전체 결측,0
음수 인구,0
행 단위 총량 오차,0
격자 장애인 목표 오차,0
전체 대상자 합계,582441
장애인 추정 합계,28538
비장애인 추정 합계,553903


## 6. 최종 결과 저장

In [6]:
output_columns = [
    "GRID_CD", "시군구", "행정동", "성별", "연령대", "행정동_장애인비율",
    disabled_col, non_disabled_col,
]
final = result[output_columns].copy()
final.to_csv(OUTPUT_FILE, encoding="utf-8-sig", index=False)

saved = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
assert len(saved) == len(final)
assert saved.isna().sum().sum() == 0
assert saved[["GRID_CD", "성별", "연령대"]].duplicated().sum() == 0
assert (saved[disabled_col] + saved[non_disabled_col]).sum() == result[base_col].sum()

display(pd.Series({
    "저장 파일": OUTPUT_FILE.name,
    "행 수": f"{len(saved):,}",
    "격자 수": f"{saved['GRID_CD'].nunique():,}",
    "장애인 추정 합계": f"{saved[disabled_col].sum():,}명",
    "비장애인 추정 합계": f"{saved[non_disabled_col].sum():,}명",
    "결측값": int(saved.isna().sum().sum()),
    "중복": int(saved[["GRID_CD", "성별", "연령대"]].duplicated().sum()),
}, name="결과").to_frame())
print(f"저장 완료: {OUTPUT_FILE}")

,결과
저장 파일,서울시_격자_100m_문화누리대상자_성연령장애별_인구수.csv
행 수,"1,452,672"
격자 수,"60,528"
장애인 추정 합계,"28,538명"
비장애인 추정 합계,"553,903명"
결측값,0
중복,0


저장 완료: C:\Users\ben20\Documents\DataScience\projects\Oracle-Project\data\processed\estimated_target_population\서울시_격자_100m_문화누리대상자_성연령장애별_인구수.csv
